In [ ]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys
import subprocess
from pathlib import Path
import json


# Local files/code
import src.data_preprocessing.image_data_exploration as img_explore
import src.data_preprocessing.text_preprocessing as text_pre
import src.data_preprocessing.text_data_exploration as text_explore
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
import src.config as config
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
import src.evaluation.visual_evaluator as visual_evaluator


In [ ]:

# example of how to chunk+vectorize text data

USDA_data_path= DATA_DIR / "EOI_Data_Pulls"
USDA_plant_sheets_path= DATA_DIR / "EOI_Data_Pulls" / "plant_sheets"

In [ ]:
# Chunk and embed the USDA data

# make the text embedder
embedder= TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")

CHUNK_METHOD= "word" # word or sentence
CHUNK_MAX_SIZE= 64
CHUNK_OVERLAP_AMOUNT= 16

# chunk and encode
records= embedder.chunk_and_encode(text, CHUNK_METHOD, CHUNK_MAX_SIZE, CHUNK_OVERLAP_AMOUNT)
# This takes ~30sec - 1.5min to run
# parse the USDA plant sheets, chunk them, vectorize them, and make organized records
# Also chunk/vectorize USDA json files that have relevant plant sheets
# NOTE: the PDF parsing library loves to spit out annoying logs
#       I've tried to supress them a million ways, but could not get it to stop
#       The logs are annoying but superficial
records= embed_USDA_data(
    config.Data.USDA.ROOT,
    embedder,
    CHUNK_METHOD,
    CHUNK_MAX_SIZE,
    CHUNK_OVERLAP_AMOUNT
)

# review records
for record in records:
    Logger.info(f"\n{record}")



In [ ]:
# Example of one of the records for one chunk of a plant
Logger.info(f"records type: {type(records)}")
Logger.info(f"records len: {len(records)}")
Logger.info(f"Example of a pdf chunk record: {records[0]}")
Logger.info(f"Example of a json record: \n\n{records[-1]}")

In [ ]:


# preprocess (not tokenize) the training, testing, and validation datasets

plant_expert_vqa_TRAIN= text_pre.load_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
plant_expert_vqa_TEST= text_pre.load_csv(config.Data.PlantExpertVQA.TEST_FILE)
plant_expert_vqa_VAL= text_pre.load_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)

# training
text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)


In [ ]:
# Testing the tokenizer and showing how to use it

# intialize the tokenizer
tokenizer= TextTokenizer("distilbert-base-uncased", 128)

# example of the passing in a single string to tokenize
inputs, attention_mask= tokenizer.encode_text("hello")
Logger.info(inputs)

# how to decode a single tensor output
output= tokenizer.decode(inputs)
Logger.info(output)

# encoding a list of strings
inputs, attention_mask= tokenizer.encode_text(["hello", "this is a test", "i am putting in multiple strings"])
Logger.info(inputs)

# decoding a list of tensors
output= tokenizer.batch_decode(inputs)
Logger.info(output)

# encoding a column of data
questions= plant_expert_vqa_TRAIN["question_text"]
inputs, attention_mask= tokenizer.encode_text(questions)

# decoding a all of those tensors
output= tokenizer.batch_decode(inputs)
Logger.info(f"first few decoded Tensors: \n\n{output[:10]}")


In [ ]:
# Most of the images are a small photo pasted onto a large blank canvas, so we crop before resizing
img_explore.show_padding(plant_expert_vqa_TRAIN, plant_expert_vqa_path, count=8)

# The cropped image next to the two channels the mask branch receives
img_explore.show_cropped_and_mask(plant_expert_vqa_TRAIN, plant_expert_vqa_path, count=3)

In [ ]:
# Adding on macbook - will need to retest on workstation later tonight due to GPU acceleration
traits = ["crop", "disease", "severity"]

classes = {}
for t in traits:
    names = sorted(plant_expert_vqa_TRAIN[t].unique())  # adds names of plants
    classes[t] = names + ["unknown"]  # adds unknown species

train_img = VisualModel.one_row_per_image(plant_expert_vqa_TRAIN, traits)
val_img = VisualModel.one_row_per_image(plant_expert_vqa_VAL, traits)
test_img = VisualModel.one_row_per_image(plant_expert_vqa_TEST, traits)
classes = VisualModel.build_classes(train_img, traits)

train_ds = VisualModel.make_dataset(train_img, classes, config.Data.PlantExpertVQA.ROOT, training=True)
val_ds = VisualModel.make_dataset(val_img, classes, config.Data.PlantExpertVQA.ROOT)

# every image shows up once per question, and those rows do not always agree on a label
img_explore.show_label_conflicts(plant_expert_vqa_TRAIN, traits)

In [ ]:
# disease has a long tail of classes with very few images
support = img_explore.show_class_support(train_img, traits)

In [ ]:
visual_model = VisualModel(classes=classes)
visual_model.build()
visual_model.compile()
history_frozen = visual_model.fit(train_ds, val_ds, epochs=20)
visual_model.unfreeze()
history_tuned = visual_model.fit(train_ds, val_ds, epochs=5)

visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease"])

In [ ]:
import json
Path("./models/visual_models/9_19_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/9_19.keras")

In [ ]:
MODEL_DIR = Path("./models/visual_models")

vm = VisualModel.load(MODEL_DIR / "9_19.keras", MODEL_DIR/"9_19_classes.json")

In [ ]:
test_table, test_predictions = visual_evaluator.evaluate(vm, plant_expert_vqa_TEST, config.Data.PlantExpertVQA.ROOT, name="Test_VisualMoel")
test_table